## 3. Feature engineering

Turning the loaded database into a model-ready dataset, with the feature logic sql/feature_engineering.sql

Steps:
1. Creating the model_features VIEW from the SQL file
2. Validating the WINDOW functions
3. Checking for target leakage (rolling averages exclude the current row)
4. Auditing nulls
5. Turning it into a table & exporting


### 1. Setup

In [1]:
import sqlite3   
from pathlib import Path  
import pandas as pd   
import numpy as np   

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

CWD=Path.cwd()
PROJECT_ROOT=CWD.parent if CWD.name=="notebooks" else CWD  

SQL_DIR=PROJECT_ROOT/"sql"
DB_PATH=PROJECT_ROOT/"database"/"energy.db"
PROCESSED_DIR=PROJECT_ROOT/"data"/"processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

assert DB_PATH.exists(), f"energy.db not found at {DB_PATH}"

conn=sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys=ON;")

for t in ["demand_readings", "calendar_info", "time_periods", "weather_hourly"]:
    n=conn.execute(f"SELECT COUNT(*) FROM {t};").fetchone()[0]
    print(f" {t:<20} {n:>10,}")


 demand_readings          52,608
 calendar_info             1,103
 time_periods                 50
 weather_hourly           26,304


### 2. Creating the view

Reading feature_engineering.sql file and handing it to executescript

In [2]:
feature_sql=(SQL_DIR / "feature_engineering.sql").read_text(encoding="utf-8")
conn.executescript(feature_sql)
conn.commit()

# Creating an index on the bank holiday lookup. Instead of a full table scan every time, the calendar_extras CTE searches the calendar_info for the 
# nearest bank holiday to each date
conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_calendar_bank_holiday
    ON calendar_info (is_bank_holiday, calendar_date);
""")
conn.commit()

n_rows=conn.execute("SELECT COUNT(*) FROM model_features;").fetchone()[0]
sample=pd.read_sql_query("SELECT * FROM model_features LIMIT 5;",conn)


print(f"model_features view created: {n_rows:,} rows x {len(sample.columns)} columns")
print("\nColumns:")
for i,c in enumerate(sample.columns,1):
    print(f" {i:>2}. {c}")


model_features view created: 52,602 rows x 49 columns

Columns:
  1. settlement_date
  2. settlement_period
  3. time_label
  4. tsd_mw
  5. hour
  6. hour_decimal
  7. period_of_day
  8. is_peak
  9. year
 10. month
 11. day_of_week
 12. day_name
 13. day_of_year
 14. season
 15. is_weekend
 16. is_bank_holiday
 17. is_school_holiday
 18. is_working_day
 19. is_clock_change_day
 20. days_since_bank_holiday
 21. days_to_bank_holiday
 22. week_slot
 23. is_working_peak
 24. temperature_c
 25. temperature_lag_1d
 26. temperature_roll_3d_avg
 27. wind_speed
 28. humidity_pct
 29. precipitation_mm
 30. cloud_cover_pct
 31. solar_radiation
 32. heating_degrees
 33. cooling_degrees
 34. demand_lag_1d
 35. demand_lag_2d
 36. demand_lag_7d
 37. demand_lag_14d
 38. demand_roll_7d_avg
 39. demand_roll_28d_avg
 40. demand_roll_7d_min
 41. demand_roll_7d_max
 42. demand_roll_7d_range
 43. demand_lag_1d_vs_avg
 44. demand_wow_change
 45. embedded_wind_mw
 46. embedded_solar_mw
 47. pump_storage_mw


### 3 Validation

Cheking if demand_lag_1d actually holds yesterday's value, by joining the view back to the raw demand_readings table (on yesterday's date), then checking if the 2 columns agree.

In [3]:
def check_lag(lag_days:int, column:str) -> None:
    #Verifying a lag column equals the raw value from N days earlier, same period
    q=f"""
        SELECT f.settlement_date,
                f.settlement_period,
                f.{column}  AS from_window_function,
                d.tsd_mw    AS actual_value_n_days_ago
        FROM model_features f
        JOIN demand_readings d
            ON d.settlement_date = date(f.settlement_date, '-{lag_days} day')
            AND d.settlement_period = f.settlement_period
        WHERE f.{column} IS NOT NULL;
    
    """

    t=pd.read_sql_query(q,conn)
    diff=(t["from_window_function"] - t["actual_value_n_days_ago"]).abs()
    ok=diff.max() < 1e-9

    print(f" {column:<18} rows checked: {len(t):>7,} "
          f"max difference: {diff.max():.2e}  {'PASS' if ok else 'FAIL'}")
    assert ok, f"{column} does not match the raw value {lag_days} days back"

print("LAG VALIDATION")
check_lag(1, "demand_lag_1d")
check_lag(2, "demand_lag_2d")
check_lag(7, "demand_lag_7d")
check_lag(14, "demand_lag_14d")
print("\nAll lag columns point at the correct historical row")





LAG VALIDATION
 demand_lag_1d      rows checked:  52,548 max difference: 0.00e+00  PASS
 demand_lag_2d      rows checked:  52,500 max difference: 0.00e+00  PASS
 demand_lag_7d      rows checked:  52,260 max difference: 0.00e+00  PASS
 demand_lag_14d     rows checked:  51,924 max difference: 0.00e+00  PASS

All lag columns point at the correct historical row


### 4. Leakage check

A rolling average is meant to summarise the recent past. If it accidentally includes the value
we are trying to predict, the model gets to see the answer. 

The test below reconstructs the same calculation in pandas using a complete calendar grid and
`.shift(1)` -->  one shift equals one calendar day rather than one row. 

In [4]:
raw=pd.read_sql_query(
    "SELECT settlement_date, settlement_period, tsd_mw FROM demand_readings;",conn
)

#Reindexing each period onto a complete date grid --> .shift(1) means
#"one calendar day back", not "one row back"

raw["d"]=pd.to_datetime(raw["settlement_date"])
full=pd.date_range(raw["d"].min(), raw["d"].max(), freq="D")

clean_parts=[]
leaky_parts=[]

for p,g in raw.groupby("settlement_period"):
    s=g.set_index("d")["tsd_mw"].reindex(full)
    clean=s.shift(1).rolling(7, min_periods=1).mean()
    leaky=s.rolling(7, min_periods=1).mean()
    clean_parts.append(pd.DataFrame({
        "settlement_date": full.strftime("%Y-%m-%d"),
        "settlement_period": p,
        "expected_clean": clean.values,
        "leaky": leaky.values,
    }))

expected=pd.concat(clean_parts, ignore_index=True)

from_sql=pd.read_sql_query(
    "SELECT settlement_date, settlement_period, demand_roll_7d_avg FROM model_features;",conn
)

merged=(
    expected.merge(from_sql, on=["settlement_date", "settlement_period"])
    .dropna(subset=["expected_clean", "demand_roll_7d_avg"])
)

error=(merged["expected_clean"] - merged["demand_roll_7d_avg"]).abs()
leak_gap=(merged["leaky"] - merged["expected_clean"]).abs()

print("Leakage check - demand_roll_7d_avg")
print("-"*40)
print(f"Rows compared: {len(merged):>10,}")
print(f"Max difference vs leakage-free version: {error.max():>10.2e}")
print(f"Verdict: " 
      f"{'PASS - no leakage' if error.max() <1e-6 else 'FAIL'}")

assert error.max() <1e-6, "Rolling average is leaking the current row"




Leakage check - demand_roll_7d_avg
----------------------------------------
Rows compared:     52,554
Max difference vs leakage-free version:   0.00e+00
Verdict: PASS - no leakage


### 5. Null audit & starting period

Demand_lag_14d is NULL for the first 14 days in the data set.
Demand_roll_28d_avg is supposed to give the previous 28 days' average (SQL will give the average of how many days has data, for example demand_roll_28d on the 3rd day in the data set, will produce day 1's and day 2's average ).

A model trained across all of that wil learn from a feature that changes its character partway through.

The fix for both is to start the modelling set 28 days in.


In [5]:
null_summary=pd.read_sql_query(""" 
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN demand_lag_1d IS NULL THEN 1 ELSE 0 END) AS null_lag_1d,
        SUM(CASE WHEN demand_lag_7d IS NULL THEN 1 ELSE 0 END) AS null_lag_7d,
        SUM(CASE WHEN demand_lag_14d IS NULL THEN 1 ELSE 0 END) AS null_lag_14d,
        SUM(CASE WHEN demand_roll_7d_avg IS NULL THEN 1 ELSE 0 END) AS null_roll_7d,
        SUM(CASE WHEN demand_roll_28d_avg IS NULL THEN 1 ELSE 0 END) AS null_roll_28d,
        SUM(CASE WHEN temperature_c IS NULL THEN 1 ELSE 0 END) AS null_temperature,
        SUM(CASE WHEN tsd_mw IS NULL THEN 1 ELSE 0 END) AS null_target
    FROM model_features;
""",conn)

print("NULL COUNTS")
print(null_summary.T.rename(columns={0: "count"}).to_string())

#NULLs should equal (lag days x 48 periods + (3 March clock-change days x 2 periods))
print(f"\nExpected NULLs for lag_14d: 14x48 + (3 March clock-change days x 2 periods)= {14*48+6}")
print("-"*40)
print("Weather coverage:")

gaps=pd.read_sql_query("""
    SELECT settlement_date, COUNT(*) AS periods_without_weather
    FROM model_features
    WHERE temperature_c IS NULL
    GROUP BY settlement_date
    ORDER BY settlement_date;
""",conn)

print(f"  {len(gaps)} date(s) have missing weather:")
print(gaps.head(20).to_string(index=False))






NULL COUNTS
                  count
total_rows        52602
null_lag_1d          54
null_lag_7d         342
null_lag_14d        678
null_roll_7d         48
null_roll_28d        48
null_temperature      0
null_target           0

Expected NULLs for lag_14d: 14x48 + (3 March clock-change days x 2 periods)= 678
----------------------------------------
Weather coverage:
  0 date(s) have missing weather:
Empty DataFrame
Columns: [settlement_date, periods_without_weather]
Index: []


In [6]:
first_date=pd.read_sql_query(
    "SELECT MIN(settlement_date) AS d FROM model_features;",conn
)["d"].iloc[0]

MODEL_START_DATE=(pd.Timestamp(first_date) + pd.Timedelta(days=28)).strftime("%Y-%m-%d")

counts=pd.read_sql_query(f"""
    SELECT
        COUNT(*) AS all_rows,
        SUM(CASE WHEN settlement_date >= '{MODEL_START_DATE}' THEN 1 ELSE 0 END) AS usable_rows
    FROM model_features;
""", conn)

all_rows=int(counts["all_rows"].iloc[0])
usable_rows=int(counts["usable_rows"].iloc[0])

print(f"Data starts: {first_date}")
print(f"Modelling starts: {MODEL_START_DATE} (28-days in)")
print(f"Rows discarded: {all_rows-usable_rows:,}")
print(f"Rows retained: {usable_rows:,} ({usable_rows/all_rows:.1%})")


Data starts: 2023-01-01
Modelling starts: 2023-01-29 (28-days in)
Rows discarded: 1,344
Rows retained: 51,258 (97.4%)


### 6. Feature correlations in the training period

In [7]:
features=pd.read_sql_query(
    f"SELECT * FROM model_features WHERE settlement_date>= '{MODEL_START_DATE}';",conn

)

print(f"Loaded {len(features):,} rows x {features.shape[1]} columns")

# Inspecting feature relationships using the training period only
training_features = features.loc[
    pd.to_datetime(features["settlement_date"])
    < pd.Timestamp("2025-01-01")
].copy()

numeric = training_features.select_dtypes(include=[np.number])

print(
    f"Correlation analysis uses "
    f"{len(training_features):,} training-period rows."
)
corr=(
    numeric.corr(numeric_only=True)["tsd_mw"]
    .drop("tsd_mw")
    .sort_values(key=abs, ascending=False)
)

print("\nCorrelation with tsd_mw (top 15 absolute value):")
print(corr.head(15).round(3).to_string())
print("\nWeakest 5:")
print(corr.tail(5).round(3).to_string())




Loaded 51,258 rows x 49 columns
Correlation analysis uses 33,740 training-period rows.

Correlation with tsd_mw (top 15 absolute value):
nd_mw                   0.983
underlying_demand_mw    0.894
demand_lag_1d           0.891
demand_roll_7d_avg      0.887
demand_lag_7d           0.883
demand_roll_7d_max      0.872
demand_roll_7d_min      0.844
demand_roll_28d_avg     0.843
demand_lag_14d          0.829
demand_lag_2d           0.793
hour                    0.414
settlement_period       0.413
hour_decimal            0.413
demand_roll_7d_range    0.387
heating_degrees         0.384

Weakest 5:
humidity_pct           0.029
is_clock_change_day   -0.027
day_of_year           -0.020
month                 -0.018
demand_wow_change     -0.004


nd_mw correlates 0.981 with tsd_mw. It's a different meter reading of the same physical quantity at the same instant. It cannot enter the model, but not because of the high correlation: when the model wants to predict the next half hour's TSD, it doesn't have the next half hours's ND either, it hasn't been measured yet. The model would be fed with a value that doesn't exist at prediction time.

underlying_demand_mw correlates 0.876 with tsd_mw. THe feature cannot enter the model, as it was defined as tsd_mw + embedded_wind + embedded_solar. The model would be able to recover tsd_mw by substracting the 2 embedded columns. --> leakage



In [8]:
#DEfining columns that won't be used as model inputs
LEAKY_COLUMNS= [
    "tsd_mw",
    "underlying_demand_mw",
    "nd_mw",
]

# DEfining identifiers (these are not features)
ID_COLUMNS=["settlement_date",
            "time_label",
            "day_name",
            "period_of_day",
            "season",]

#Excluding features and identifiers
model_ready=[c for c in features.columns if c not in LEAKY_COLUMNS + ID_COLUMNS]

print(f"Columns in view: {features.shape[1]}")
print(f"Excluded leakage: {LEAKY_COLUMNS}")
print(f"Exclided identifiers: {ID_COLUMNS}")
print(f"\nModel-ready feature count: {len(model_ready)}")
print("\nFeatures:")
for c in model_ready:
    print(f" - {c}")





Columns in view: 49
Excluded leakage: ['tsd_mw', 'underlying_demand_mw', 'nd_mw']
Exclided identifiers: ['settlement_date', 'time_label', 'day_name', 'period_of_day', 'season']

Model-ready feature count: 41

Features:
 - settlement_period
 - hour
 - hour_decimal
 - is_peak
 - year
 - month
 - day_of_week
 - day_of_year
 - is_weekend
 - is_bank_holiday
 - is_school_holiday
 - is_working_day
 - is_clock_change_day
 - days_since_bank_holiday
 - days_to_bank_holiday
 - week_slot
 - is_working_peak
 - temperature_c
 - temperature_lag_1d
 - temperature_roll_3d_avg
 - wind_speed
 - humidity_pct
 - precipitation_mm
 - cloud_cover_pct
 - solar_radiation
 - heating_degrees
 - cooling_degrees
 - demand_lag_1d
 - demand_lag_2d
 - demand_lag_7d
 - demand_lag_14d
 - demand_roll_7d_avg
 - demand_roll_28d_avg
 - demand_roll_7d_min
 - demand_roll_7d_max
 - demand_roll_7d_range
 - demand_lag_1d_vs_avg
 - demand_wow_change
 - embedded_wind_mw
 - embedded_solar_mw
 - pump_storage_mw


### 8. Creating table & export

In [9]:
conn.execute("DROP TABLE IF EXISTS model_features_tbl;")
conn.execute(f"""
    CREATE TABLE model_features_tbl AS
    SELECT * FROM model_features
    WHERE settlement_date >='{MODEL_START_DATE}';
""")
conn.execute(""" 
    CREATE INDEX IF NOT EXISTS idx_mf_date
    ON model_features_tbl (settlement_date, settlement_period);
""")
conn.commit()

n=conn.execute("SELECT COUNT(*) FROM model_features_tbl;").fetchone()[0]
print(f"model_features_tbl created: {n:,} rows")

#Exporting a file copy
csv_path=PROCESSED_DIR / "model_features.csv"
features.to_csv(csv_path, index=False)
print(f"\nCSV written: {csv_path.name} ({csv_path.stat().st_size / 1_000_000:.1f} MB)")

#Writing it into parquet (smaller and preserves dtypes)
pq_path=PROCESSED_DIR / "model_features.parquet"
features.to_parquet(pq_path, index=False)

print(f"Parquet written: {pq_path.name} ({pq_path.stat().st_size / 1_000_000:.1f} MB)")

model_features_tbl created: 51,258 rows

CSV written: model_features.csv (15.9 MB)
Parquet written: model_features.parquet (3.6 MB)


In [10]:
conn.commit()
conn.close()
